# BB84-Protocol

The BB84 protocol was first introduced by Bennett and Brassard in 1984 (hence the name) and it suggests the exchange of information between two parties discretely and using Quantum logic, by eliminating the compromise of the information to a third party who is not a part of the information exchange i.e. an eavesdropper.

### The logic-

1. We have two parties- Alice and Bob. Alice wants to send a piece of information to Bob. She can make use of: Z bases ($|0\rangle$ for 0 or $|1\rangle$ for 1) as computational bases and X bases ($\frac{1}{\sqrt 2}(|0\rangle + |1\rangle)$ for 0 or $\frac{1}{\sqrt 2}(|0\rangle - |1\rangle)$ for 1) as Hadamard Bases.

2. She chooses random classical bits (0 or 1) and encodes it using her choice of X bases to form her qubits which she then sends to Bob

3. Bob then chooses random measurement bases Z or X to measure the qubits sent by Alice.

4. After measurement, Alice and Bob then publicly announce the measurement bases they used but NOT the bit values.

5. They discard the bases that do not match and keep the ones that match.

6. If Eve (a third party eavesdropper) was sabotaging the exchange protocol, she will be choosing wrong bases randomly, thereby affecting the exchange process and producing detectable errors.

### Why its secure?

If Eve tries to sabotage the process, she will induce errors because she will be choosing the measurement bases on her own which she can do with a 50% probability. The errors become therefore measurable. Furthermore, Eve cannot copy the information that Alice has been sending due to No Cloning Theorem.

The security is highlighted through the errors. Imagine both Alice and Bob use same measurement bases. Whatever qubit Alice is sending to be received by Bob, must be intact. When Eve is present and she receives the qubit from Alice, she also randomly chooses the measurement bases and cannot detect with 100% probability what Alice has sent to her. So she also sends in a qubit that she has prepared from her end to Bob in order to remain undetected. When Bob measures it with the same measurement basis as Alice, he will get the corrupted qubit 50% of the time. It is only during the time when both Alice and Bob sacrifice a part of their key to compare the measurement bases, does the presence of Eve come out. Due to the corrupted qubit, Bob will have a different qubit as compared to Alice if they chose the same measurement bases. 

The overall error rate is around 25% for this process. An error rate close to this highlights the presence of an eavesdropper and thus they can abort the protocol.

### Features of this program-

1. It asks for the user to mention the number of qubits to be generated.
2. One can toggle between simulating with or without an eavesdropper.
3. The circuit can also be visualized alongwith the shared key between Alice and Bob

In [8]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import random

n=int(input("Hi Alice! How many qubits do you want to prepare?"))
choice=str(input("Do you want to simulate with the presence of Eve?:(Yes/No)"))

def bb84(n, choice):
    #Preparation of qubits by Alice
    alice_bases=[]
    alice_bits=[]
    for i in range(n):
        alice_bases.append(random.choice(['Z','X']))
        alice_bits.append(random.randint(0,1))

    if choice=='Yes' or choice.lower()=='yes':
        #Presence of Eve
        eve_bases=[]
        eve_bits=[]
        for i in range(n):
            eve_bases.append(random.choice(['Z','X']))
            if eve_bases[i]==alice_bases[i]:
                eve_bits.append(alice_bits[i])
            else:
                eve_bits.append(random.randint(0,1))
                
        circuit=QuantumCircuit(n,n)
        
        for i in range(n):
            if eve_bits[i]==1:
                circuit.x(i)
            if eve_bases[i]=='X':
                circuit.h(i)
        bob_bases=[]
        for i in range(n):
            bob_bases.append(random.choice(['Z','X']))
            if bob_bases[i]=='X':
                circuit.h(i)
        for i in range(n):
            circuit.measure(i,i)

        simulator = AerSimulator()

        result = simulator.run(circuit, shots=1).result()

        counts = result.get_counts()

        bob_bits=list(counts.keys())[0][::-1]

        #Comparison

        shared_key=[]

        for i in range(n):
            if alice_bases[i]==bob_bases[i]:
                shared_key.append(alice_bits[i])
        bob_key = []
        for i in range(n):
            if alice_bases[i] == bob_bases[i]:
                bob_key.append(int(bob_bits[i]))

        errors = 0
        for i in range(len(shared_key)):
            if shared_key[i] != bob_key[i]:
                errors += 1

        error_rate = errors / len(shared_key) if len(shared_key) > 0 else 0
        print(f"Errors: {errors} out of {len(shared_key)}")
        print(f"Error rate: {error_rate:.2%}")

        if error_rate > 0.1:
            print("Eve detected! Key compromised. Abort!")
        else:
            print("Channel appears secure.")

        print("Alice's key:", shared_key)
        print("Bob's key:", bob_key)
        print(circuit.draw())
        
    elif choice=='No' or choice.lower()=='no':
        #Absence of Eve
        circuit=QuantumCircuit(n,n)
        
        for i in range(n):
            if alice_bits[i]==1:
                circuit.x(i)
            if alice_bases[i]=='X':
                circuit.h(i)

        bob_bases=[]
        for i in range(n):
            bob_bases.append(random.choice(['Z','X']))
            if bob_bases[i]=='X':
                circuit.h(i)
        for i in range(n):
            circuit.measure(i,i)
        
        simulator = AerSimulator()

        result = simulator.run(circuit, shots=1).result()

        counts = result.get_counts()

        bob_bits=list(counts.keys())[0][::-1]

        #Comparison
        shared_key=[]
        for i in range(n):
            if alice_bases[i]==bob_bases[i]:
                shared_key.append(alice_bits[i])
        bob_key = []
        for i in range(n):
            if alice_bases[i] == bob_bases[i]:
                bob_key.append(int(bob_bits[i]))
            
        print("Alice's key:", shared_key)
        print("Bob's key:", bob_key)
        print(circuit.draw()) 

    else:
        print("Invalid input!")

protocol=bb84(n, choice)

Hi Alice! How many qubits do you want to prepare? 50
Do you want to simulate with the presence of Eve?:(Yes/No) yes


Errors: 4 out of 18
Error rate: 22.22%
Eve detected! Key compromised. Abort!
Alice's key: [1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1]
Bob's key: [1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1]
      ┌───┐                              ┌─┐                                 »
 q_0: ┤ H ├──────────────────────────────┤M├─────────────────────────────────»
      ├───┤                              └╥┘┌─┐                              »
 q_1: ┤ X ├───────────────────────────────╫─┤M├──────────────────────────────»
      ├───┤┌───┐                          ║ └╥┘                              »
 q_2: ┤ X ├┤ H ├──────────────────────────╫──╫───────────────────────────────»
      ├───┤└───┘                          ║  ║ ┌─┐                           »
 q_3: ┤ X ├───────────────────────────────╫──╫─┤M├───────────────────────────»
      ├───┤┌───┐                          ║  ║ └╥┘                           »
 q_4: ┤ H ├┤ H ├──────────────────────────╫──╫──╫────────────────────────────»

After running for both the presence and absence of an eavesdropper, we get a 0% error rate in the absence of an eavesdropper when the communication is slowly existing between Alice and Bob.

However, for the case where we have an eavesdropper present, we get an error rate of near about 25% due to which the protocol is aborted.